In [ ]:
import numpy as np
from scipy.spatial import distance_matrix
from sklearn.preprocessing import StandardScaler

def calculate_distance_sum(p1, p2):
    # Calculate the distance matrix between corresponding points on P1 and P2
    # dist = distance_matrix(p1, p2)
    dist = np.linalg.norm(p1 - p2, axis=1)
    print("dist", dist)

    # Calculate the sum of squared distances along each row (corresponding points on P1)
    row_sums = np.sum(dist ** 2, axis=0)

    return row_sums

def optimize_registration(p1, p2):
    # Initialize translation vector and angle of rotation
    T, S, theta = 0, 0, 0
    momentum = 0.01

    # Use gradient descent to minimize the distance sum
    for i in range(100):  # number of iterations
        # Calculate the distance matrix with current transformation
        dist = calculate_distance_sum(p1 + [T, S], p2 + [np.cos(theta), np.sin(theta)])
        
        print("dist", dist)

        # Compute gradients (not shown here, but can be computed using finite differences)
        grad_T = -np.sum(dist ** 2) / len(p1)
        grad_S = -np.sum(dist ** 2) / len(p1)
        grad_theta = -np.sum(dist ** 2 * np.array([np.sin(theta), np.cos(theta)]) * [np.cos(theta), -np.sin(theta)]) / len(p1)

        # Update parameters
        T += 0.01 * grad_T * momentum
        S += 0.01 * grad_S * momentum
        theta += 0.01 * grad_theta * momentum 

    return T, S, theta

# Example usage:
p1 = [(x, y) for x, y in enumerate(np.random.rand(100))]
p2 = [(x + np.cos(theta) * 10, y + np.sin(theta) * 10) for (x, y), theta in zip(p1, [np.pi / 4] * len(p1))]

p1 = np.array(p1)
p2 = np.array(p2)

T, S, theta = optimize_registration(p1, p2)
print(T, S, theta)

# NCC

In [ ]:
import numpy as np
from scipy.signal import correlate2d

def normalized_cross_correlation(template, image):
    """
    Perform Normalized Cross-Correlation (NCC) to align a template with an image.

    Args:
        template (ndarray): The template image (smaller image).
        image (ndarray): The reference image (larger image).

    Returns:
        tuple: (y_offset, x_offset) - The offsets to align the template with the image.
    """
    # Ensure the template and image are in float format
    template = template.astype(np.float32)
    image = image.astype(np.float32)

    # Subtract the mean from the template and image
    template_mean = template - np.mean(template)
    image_mean = image - np.mean(image)

    # Compute the normalized cross-correlation
    ncc_map = correlate2d(image_mean, template_mean, mode='valid')

    # Normalize the NCC map
    template_norm = np.sqrt(np.sum(template_mean ** 2))
    image_norm = np.sqrt(correlate2d(image_mean ** 2, np.ones(template.shape), mode='valid'))
    ncc_map /= (template_norm * image_norm + 1e-5)  # Add small value to avoid division by zero

    # Find the location of the maximum NCC value
    y_offset, x_offset = np.unravel_index(np.argmax(ncc_map), ncc_map.shape)

    return y_offset, x_offset

# Example usage
if __name__ == "__main__":
    import cv2

    # Load the template and image (convert to grayscale)
    template = cv2.imread("template.png", cv2.IMREAD_GRAYSCALE)
    image = cv2.imread("image.png", cv2.IMREAD_GRAYSCALE)

    if template is None or image is None:
        print("Error: Could not load images.")
    else:
        # Perform NCC
        y_offset, x_offset = normalized_cross_correlation(template, image)
        print(f"Template matched at offset: (y: {y_offset}, x: {x_offset})")